# 03 pandas / numpy 量化够用部分

## 3.1 本章要解决什么问题

本章只补齐后续章节会反复用到的 pandas / NumPy 技能，不展开成通用 pandas 教程。你需要能看懂并亲手写出这些动作：

- 长表与宽表：`date, code, close` 面板和 `date x code` 矩阵怎么互转。
- 日期解析、排序与去重：所有时序计算都必须先确认时间顺序。
- 分组滚动：每只资产内部计算收益、动量、波动率。
- `pivot` 矩阵与 `pct_change` 收益率：给后续回测和组合权重使用。
- 对齐与缺失值：不同资产、不同日期之间先对齐，再计算。
- 向量化分数与权重：把横截面分数转成权重矩阵，避免逐行手工循环。

## 3.2 本章位置

第 02 章已经确认项目环境和缓存数据能读取。本章先把 `data/sample/` 里的真实 ETF 缓存当作已存在的数据来练手；第 04 章再解释这些缓存如何从 AKShare 获取、标准化并保存。

## 3.3 输入与输出

- 输入：`data/sample/prices.parquet`、`data/sample/assets.parquet`、`data/sample/calendar.parquet`。
- 输出：本章在内存中得到 `features`、`close_matrix`、`returns_matrix`、`score_matrix`、`weights_matrix`。为了不覆盖后续章节产物，本章不写入结果文件。

## 3.4 练习方式

先用很小的手写列表理解原则，再用 pandas / NumPy 在真实 ETF 样本上复现同一套动作。


In [1]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "lib").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find pyquant-roadmap project root from current working directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

pd.Series({"project_root": ".", "python": sys.version.split()[0]}, name="value")


project_root          .
python          3.11.15
Name: value, dtype: object

## 3.5 先不用 pandas：一张最小长表

量化面板数据最常见的原始形态是长表：每一行是某只资产在某个交易日的一条记录。先不用 pandas，看清楚它本质上只是带 `date` 和 `code` 的记录列表。


In [2]:
tiny_rows = [
    {"date": "2023-01-03", "code": "ETF_A", "close": 102.0},
    {"date": "2023-01-02", "code": "ETF_A", "close": 100.0},
    {"date": "2023-01-04", "code": "ETF_A", "close": 101.0},
    {"date": "2023-01-02", "code": "ETF_B", "close": 50.0},
    {"date": "2023-01-04", "code": "ETF_B", "close": 49.0},
]

tiny_rows


[{'date': '2023-01-03', 'code': 'ETF_A', 'close': 102.0},
 {'date': '2023-01-02', 'code': 'ETF_A', 'close': 100.0},
 {'date': '2023-01-04', 'code': 'ETF_A', 'close': 101.0},
 {'date': '2023-01-02', 'code': 'ETF_B', 'close': 50.0},
 {'date': '2023-01-04', 'code': 'ETF_B', 'close': 49.0}]

### 3.5.1 日期解析与排序

时序计算不能相信原始行顺序。第一步是把字符串日期解析成真正的日期，再按 `code, date` 排序，这样每只资产内部才有正确的前后关系。


In [3]:
from datetime import datetime


def parse_and_sort_rows(rows: list[dict]) -> list[dict]:
    parsed = []
    for row in rows:
        item = row.copy()
        item["date"] = datetime.strptime(item["date"], "%Y-%m-%d").date()
        parsed.append(item)
    return sorted(parsed, key=lambda item: (item["code"], item["date"]))


tiny_sorted = parse_and_sort_rows(tiny_rows)
tiny_sorted


[{'date': datetime.date(2023, 1, 2), 'code': 'ETF_A', 'close': 100.0},
 {'date': datetime.date(2023, 1, 3), 'code': 'ETF_A', 'close': 102.0},
 {'date': datetime.date(2023, 1, 4), 'code': 'ETF_A', 'close': 101.0},
 {'date': datetime.date(2023, 1, 2), 'code': 'ETF_B', 'close': 50.0},
 {'date': datetime.date(2023, 1, 4), 'code': 'ETF_B', 'close': 49.0}]

### 3.5.2 手写长表转宽表

宽表矩阵的行是日期，列是资产。组合回测、收益矩阵、权重矩阵都更适合这种形状。下面故意保留 `ETF_B` 在 `2023-01-03` 的缺口，用来观察对齐问题。


In [4]:
def make_manual_close_matrix(rows: list[dict]) -> tuple[list, list, dict]:
    dates = sorted({row["date"] for row in rows})
    codes = sorted({row["code"] for row in rows})
    matrix = {date: {code: None for code in codes} for date in dates}
    for row in rows:
        matrix[row["date"]][row["code"]] = row["close"]
    return dates, codes, matrix


tiny_dates, tiny_codes, tiny_close_matrix = make_manual_close_matrix(tiny_sorted)
tiny_close_matrix


{datetime.date(2023, 1, 2): {'ETF_A': 100.0, 'ETF_B': 50.0},
 datetime.date(2023, 1, 3): {'ETF_A': 102.0, 'ETF_B': None},
 datetime.date(2023, 1, 4): {'ETF_A': 101.0, 'ETF_B': 49.0}}

### 3.5.3 手写收益率与缺失值

`pct_change` 的公式只有一行：`今天价格 / 昨天价格 - 1`。真正容易出错的是缺失值：如果今天或昨天缺价，就不要假装自己知道收益率。


In [5]:
def manual_pct_change(values: list[float | None]) -> list[float | None]:
    returns = [None]
    for previous, current in zip(values[:-1], values[1:]):
        if previous is None or current is None or previous == 0:
            returns.append(None)
        else:
            returns.append(current / previous - 1)
    return returns


manual_returns = {}
for code_ in tiny_codes:
    closes = [tiny_close_matrix[date][code_] for date in tiny_dates]
    manual_returns[code_] = manual_pct_change(closes)

manual_returns


{'ETF_A': [None, 0.020000000000000018, -0.009803921568627416],
 'ETF_B': [None, None, None]}

### 3.5.4 手写分组滚动与权重

后续因子会大量使用“每只资产内部滚动计算，再在同一天横截面比较”。用小列表可以看出两个层次：

- 时间序列层：每只资产自己和自己的过去比较。
- 横截面层：同一天资产之间比较，得到分数和权重。


In [6]:
def rolling_mean(values: list[float | None], window: int) -> list[float | None]:
    result = []
    for index in range(len(values)):
        window_values = values[index - window + 1 : index + 1]
        if len(window_values) == window and all(value is not None for value in window_values):
            result.append(sum(window_values) / window)
        else:
            result.append(None)
    return result


manual_rolling_returns = {
    code_: rolling_mean(values, window=2)
    for code_, values in manual_returns.items()
}
manual_latest_scores = {"ETF_A": 0.8, "ETF_B": 0.2}
manual_total_score = sum(manual_latest_scores.values())
manual_weights = {code_: score / manual_total_score for code_, score in manual_latest_scores.items()}

manual_rolling_returns, manual_weights


({'ETF_A': [None, None, 0.005098039215686301], 'ETF_B': [None, None, None]},
 {'ETF_A': 0.8, 'ETF_B': 0.2})

## 3.6 用 pandas / NumPy 处理真实 ETF 缓存

pandas 负责表格索引、分组、滚动窗口、透视矩阵和缺失值处理；NumPy 负责矩阵级别的快速数值运算。后续章节不会把这些细节全部藏进 `lib/`，因为读者需要能判断计算是否对齐、是否使用了未来数据。


In [7]:
from lib.data import load_sample_assets, load_sample_calendar, load_sample_prices

assets = load_sample_assets().copy()
calendar = load_sample_calendar().copy()
prices_raw = load_sample_prices().copy()

display(assets)
display(prices_raw.head())


,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,date,code,open,high,low,close,volume,amount
0,2021-01-04,159915,2.864,2.990,2.861,2.976,2472764,7.278150e+08
1,2021-01-04,510300,4.789,4.875,4.769,4.843,5067056,2.697193e+09
2,2021-01-04,510500,5.926,6.046,5.899,6.014,3035862,2.166750e+09
3,2021-01-04,512100,2.493,2.545,2.488,2.537,1121520,1.064703e+08
4,2021-01-05,159915,2.940,2.997,2.925,2.988,2470004,7.346481e+08


### 3.6.1 日期解析、排序与基础检查

真实数据也先做同样的动作：日期转 `datetime64`、代码转字符串、按 `code, date` 排序，并确认每个 `date, code` 只有一条价格。


In [8]:
prices = prices_raw.copy()
prices["date"] = pd.to_datetime(prices["date"])
prices["code"] = prices["code"].astype(str)
prices = prices.sort_values(["code", "date"]).reset_index(drop=True)

asset_names = assets.assign(code=assets["code"].astype(str)).set_index("code")["name"]

duplicate_rows = prices.duplicated(["date", "code"]).sum()
monotonic_by_code = prices.groupby("code")["date"].apply(lambda s: s.is_monotonic_increasing)

quality_checks = pd.Series(
    {
        "rows": len(prices),
        "assets": prices["code"].nunique(),
        "start": prices["date"].min().date(),
        "end": prices["date"].max().date(),
        "duplicate_date_code_rows": int(duplicate_rows),
        "all_codes_sorted_by_date": bool(monotonic_by_code.all()),
    }
)

coverage = (
    prices.groupby("code")
    .agg(start=("date", "min"), end=("date", "max"), rows=("date", "size"))
    .join(asset_names.rename("name"))
    .loc[:, ["name", "start", "end", "rows"]]
)

assert duplicate_rows == 0
assert monotonic_by_code.all()

display(quality_checks.to_frame("value"))
display(coverage)


,value
rows,2900
assets,4
start,2021-01-04
end,2023-12-29
duplicate_date_code_rows,0
all_codes_sorted_by_date,True


,name,start,end,rows
code,,,,
159915,创业板ETF,2021-01-04,2023-12-29,725
510300,沪深300ETF,2021-01-04,2023-12-29,725
510500,中证500ETF,2021-01-04,2023-12-29,725
512100,中证1000ETF,2021-01-04,2023-12-29,725


### 3.6.2 长表：适合按资产滚动计算

长表保留了 `date` 和 `code` 两个维度，适合 `groupby("code")` 后计算每只 ETF 自己的收益、动量、波动率。


In [9]:
long_view = (
    prices.merge(asset_names.rename("name"), left_on="code", right_index=True, how="left")
    .loc[:, ["date", "code", "name", "open", "high", "low", "close", "volume"]]
)

long_view.head(8)


,date,code,name,open,high,low,close,volume
0,2021-01-04,159915,创业板ETF,2.864,2.990,2.861,2.976,2472764
1,2021-01-05,159915,创业板ETF,2.940,2.997,2.925,2.988,2470004
2,2021-01-06,159915,创业板ETF,3.000,3.030,2.953,3.003,2290438
3,2021-01-07,159915,创业板ETF,2.996,3.056,2.978,3.056,2029990
4,2021-01-08,159915,创业板ETF,3.065,3.086,3.011,3.036,1867431
5,2021-01-11,159915,创业板ETF,3.052,3.054,2.962,2.989,2068085
6,2021-01-12,159915,创业板ETF,2.980,3.076,2.954,3.076,1857681
7,2021-01-13,159915,创业板ETF,3.071,3.094,2.997,3.025,2141047


### 3.6.3 宽表：适合矩阵运算和对齐

价格矩阵的行是交易日，列是资产。后续的收益矩阵、权重矩阵、组合收益都共享这个形状。


In [10]:
close_matrix = prices.pivot(index="date", columns="code", values="close").sort_index()
close_matrix = close_matrix.rename_axis(index="date", columns="code")

close_matrix.tail()


code,159915,510300,510500,512100
date,,,,
2023-12-25,1.783,3.135,5.163,2.232
2023-12-26,1.761,3.115,5.116,2.202
2023-12-27,1.763,3.125,5.132,2.215
2023-12-28,1.833,3.209,5.235,2.262
2023-12-29,1.842,3.219,5.279,2.296


### 3.6.4 交易日对齐与缺失值

`calendar` 给出项目样本的交易日。先把价格矩阵对齐到交易日历，再检查缺失。真实项目中，停牌、上市时间不同、数据源缺口都会在这一步暴露出来。


In [11]:
calendar_dates = pd.to_datetime(calendar.loc[calendar["is_open"].eq(1), "date"]).sort_values()
close_aligned = close_matrix.reindex(calendar_dates)
close_aligned.index.name = "date"

missing_report = pd.DataFrame(
    {
        "missing_close": close_aligned.isna().sum(),
        "first_valid_date": close_aligned.apply(lambda s: s.first_valid_index()),
        "last_valid_date": close_aligned.apply(lambda s: s.last_valid_index()),
    }
).join(asset_names.rename("name"))

close_filled = close_aligned.ffill()

display(missing_report.loc[:, ["name", "missing_close", "first_valid_date", "last_valid_date"]])
display(close_filled.tail())


,name,missing_close,first_valid_date,last_valid_date
code,,,,
159915,创业板ETF,0,2021-01-04,2023-12-29
510300,沪深300ETF,0,2021-01-04,2023-12-29
510500,中证500ETF,0,2021-01-04,2023-12-29
512100,中证1000ETF,0,2021-01-04,2023-12-29


code,159915,510300,510500,512100
date,,,,
2023-12-25,1.783,3.135,5.163,2.232
2023-12-26,1.761,3.115,5.116,2.202
2023-12-27,1.763,3.125,5.132,2.215
2023-12-28,1.833,3.209,5.235,2.262
2023-12-29,1.842,3.219,5.279,2.296


### 3.6.5 `pct_change` 收益率

收益率是价格矩阵的一阶变化。这里把第一天无法计算的收益填成 `0.0`，只是为了后续矩阵相乘时形状稳定；真实绩效解释时要记得第一天并没有实际收益。


In [12]:
returns_matrix = close_filled.pct_change(fill_method=None).fillna(0.0)

single_code = "510300"
single_return_check = pd.DataFrame(
    {
        "close": close_filled[single_code].head(6),
        "ret_1d": returns_matrix[single_code].head(6),
    }
)

display(single_return_check)
display(returns_matrix.tail().round(4))


,close,ret_1d
date,,
2021-01-04,4.843,0.000000
2021-01-05,4.942,0.020442
2021-01-06,4.991,0.009915
2021-01-07,5.108,0.023442
2021-01-08,5.080,-0.005482
2021-01-11,5.017,-0.012402


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0017,0.0029,-0.0014,-0.0018
2023-12-26,-0.0123,-0.0064,-0.0091,-0.0134
2023-12-27,0.0011,0.0032,0.0031,0.0059
2023-12-28,0.0397,0.0269,0.0201,0.0212
2023-12-29,0.0049,0.0031,0.0084,0.0150


### 3.6.6 NumPy 视角：矩阵不是魔法

pandas 的 `pct_change` 背后仍然是相邻两行做数组运算。理解这一点，后面看到权重矩阵乘收益矩阵就不会觉得抽象。


In [13]:
close_values = close_filled.to_numpy(dtype=float)
np_return_values = close_values[1:] / close_values[:-1] - 1
np_returns = pd.DataFrame(
    np_return_values,
    index=close_filled.index[1:],
    columns=close_filled.columns,
)

max_abs_diff = (np_returns - returns_matrix.iloc[1:]).abs().max().max()
print(f"NumPy result equals pandas pct_change up to max abs diff: {max_abs_diff:.12f}")
np_returns.tail().round(4)


NumPy result equals pandas pct_change up to max abs diff: 0.000000000000


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0017,0.0029,-0.0014,-0.0018
2023-12-26,-0.0123,-0.0064,-0.0091,-0.0134
2023-12-27,0.0011,0.0032,0.0031,0.0059
2023-12-28,0.0397,0.0269,0.0201,0.0212
2023-12-29,0.0049,0.0031,0.0084,0.0150


## 3.7 分组滚动：收益、动量、波动率

这一步继续使用长表，因为每只 ETF 的滚动窗口必须只看自己过去的数据。`shift(1)` 的目的很重要：今天打分时只使用昨天已经能知道的 20 日动量。


In [14]:
features = prices.copy()
features["ret_1d"] = features.groupby("code")["close"].pct_change(fill_method=None)
features["mom_20_raw"] = features.groupby("code")["close"].pct_change(20, fill_method=None)
features["mom_20"] = features.groupby("code")["mom_20_raw"].shift(1)
features["vol_20"] = features.groupby("code")["ret_1d"].transform(
    lambda s: s.rolling(window=20, min_periods=20).std()
)

feature_view = (
    features.dropna(subset=["ret_1d", "mom_20", "vol_20"])
    .merge(asset_names.rename("name"), left_on="code", right_index=True, how="left")
    .loc[:, ["date", "code", "name", "close", "ret_1d", "mom_20", "vol_20"]]
)

feature_view.head(8).round({"close": 4, "ret_1d": 4, "mom_20": 4, "vol_20": 4})


,date,code,name,close,ret_1d,mom_20,vol_20
21,2021-02-02,159915,创业板ETF,3.118,0.0203,0.0269,0.0204
22,2021-02-03,159915,创业板ETF,3.106,-0.0038,0.0435,0.0205
23,2021-02-04,159915,创业板ETF,3.098,-0.0026,0.0343,0.0201
24,2021-02-05,159915,创业板ETF,3.084,-0.0045,0.0137,0.0201
25,2021-02-09,159915,创业板ETF,3.222,0.0447,0.0158,0.0219
26,2021-02-10,159915,创业板ETF,3.293,0.0220,0.0780,0.0216
27,2021-02-18,159915,创业板ETF,3.202,-0.0276,0.0705,0.0222
28,2021-02-19,159915,创业板ETF,3.170,-0.0100,0.0585,0.0222


### 3.7.1 横截面分数：同一天资产之间比较

时间序列特征算完后，打分通常发生在同一个交易日的横截面里。这里用一个简单示例：动量越高越好，波动率越低越好。


In [15]:
score_panel = features.dropna(subset=["mom_20", "vol_20"]).copy()
score_panel["mom_score"] = score_panel.groupby("date")["mom_20"].rank(pct=True)
score_panel["low_vol_score"] = score_panel.groupby("date")["vol_20"].rank(pct=True, ascending=False)
score_panel["score"] = 0.7 * score_panel["mom_score"] + 0.3 * score_panel["low_vol_score"]

latest_score_date = score_panel["date"].max()
latest_scores = (
    score_panel.loc[score_panel["date"].eq(latest_score_date), ["date", "code", "mom_20", "vol_20", "mom_score", "low_vol_score", "score"]]
    .merge(asset_names.rename("name"), left_on="code", right_index=True, how="left")
    .sort_values("score", ascending=False)
)

latest_scores.round(4)


,date,code,mom_20,vol_20,mom_score,low_vol_score,score,name
724,2023-12-29,159915,-0.0219,0.0142,1.00,0.25,0.775,创业板ETF
1449,2023-12-29,510300,-0.0222,0.0100,0.75,0.75,0.750,沪深300ETF
2174,2023-12-29,510500,-0.0300,0.0087,0.50,1.00,0.650,中证500ETF
2899,2023-12-29,512100,-0.0448,0.0108,0.25,0.50,0.325,中证1000ETF


### 3.7.2 分数矩阵与权重矩阵

把分数透视成矩阵后，可以一次性完成每天 TopN 等权。`rank(axis=1)` 表示按每一行，也就是每个交易日，在资产之间排名。


In [16]:
score_matrix = score_panel.pivot(index="date", columns="code", values="score").sort_index()

top_n = 2
score_ranks = score_matrix.rank(axis=1, ascending=False, method="first")
selected = score_ranks.le(top_n)
weights_matrix = selected.astype(float)
weights_matrix = weights_matrix.div(weights_matrix.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

latest_weights = (
    weights_matrix.loc[latest_score_date]
    .rename("weight")
    .to_frame()
    .join(asset_names.rename("name"))
    .sort_values("weight", ascending=False)
)

display(score_matrix.tail().round(3))
display(latest_weights)


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.600,0.575,1.000,0.325
2023-12-26,0.425,0.750,1.000,0.325
2023-12-27,0.425,0.925,0.825,0.325
2023-12-28,0.425,0.925,0.825,0.325
2023-12-29,0.775,0.750,0.650,0.325


,weight,name
code,,
159915,0.5,创业板ETF
510300,0.5,沪深300ETF
510500,0.0,中证500ETF
512100,0.0,中证1000ETF


### 3.7.3 对齐、滞后与矩阵相乘

权重矩阵和收益矩阵必须有相同的日期索引和资产列。组合收益用昨天收盘后已知的权重乘今天收益，所以要 `shift(1)`，这是后续所有回测代码都要遵守的基本防线。


In [17]:
aligned_weights = weights_matrix.reindex(returns_matrix.index).ffill().fillna(0.0)
portfolio_returns = (aligned_weights.shift(1).fillna(0.0) * returns_matrix).sum(axis=1)

portfolio_preview = pd.DataFrame(
    {
        "portfolio_ret": portfolio_returns,
        "selected_count": aligned_weights.gt(0).sum(axis=1),
        "weight_sum": aligned_weights.sum(axis=1),
    }
).tail(10)

portfolio_preview.round(4)


,portfolio_ret,selected_count,weight_sum
date,,,
2023-12-18,-0.0139,2,1.0
2023-12-19,-0.0004,2,1.0
2023-12-20,-0.0151,2,1.0
2023-12-21,0.0040,2,1.0
2023-12-22,-0.0060,2,1.0
2023-12-25,0.0002,2,1.0
2023-12-26,-0.0107,2,1.0
2023-12-27,0.0032,2,1.0
2023-12-28,0.0235,2,1.0


## 3.8 小练习：从价格长表到 Top1 权重

下面是一份很小的价格长表。目标是用 pandas 完成三件事：

1. 转成收盘价矩阵。
2. 计算 2 日动量，并用 `shift(1)` 表示今天只能使用昨天已经知道的动量。
3. 每天选择动量最高的一只资产，生成 Top1 等权权重矩阵。

下一格是可运行的答案脚手架。建议先把等号右边盖住，自己补一遍。


In [18]:
exercise_prices = pd.DataFrame(
    [
        ("2023-01-02", "A", 10.0),
        ("2023-01-03", "A", 11.0),
        ("2023-01-04", "A", 12.0),
        ("2023-01-05", "A", 11.0),
        ("2023-01-02", "B", 20.0),
        ("2023-01-03", "B", 19.0),
        ("2023-01-04", "B", 21.0),
        ("2023-01-05", "B", 22.0),
    ],
    columns=["date", "code", "close"],
)
exercise_prices["date"] = pd.to_datetime(exercise_prices["date"])

# TODO 1: 长表 -> 收盘价矩阵
exercise_close = exercise_prices.pivot(index="date", columns="code", values="close").sort_index()

# TODO 2: 2 日动量；shift(1) 避免今天用到今天收盘后才知道的信号
exercise_score = exercise_close.pct_change(2, fill_method=None).shift(1)

# TODO 3: 每天选分数最高的一只；没有足够历史数据的日期权重为 0
exercise_rank = exercise_score.rank(axis=1, ascending=False, method="first")
exercise_weights = exercise_rank.eq(1).astype(float).where(exercise_score.notna(), 0.0)

answer_preview = pd.concat(
    {
        "close": exercise_close,
        "score": exercise_score.round(4),
        "weight": exercise_weights,
    },
    axis=1,
)
answer_preview


close       score       weight     
code           A     B     A     B      A    B
date                                          
2023-01-02  10.0  20.0   NaN   NaN    0.0  0.0
2023-01-03  11.0  19.0   NaN   NaN    0.0  0.0
2023-01-04  12.0  21.0   NaN   NaN    0.0  0.0
2023-01-05  11.0  22.0   0.2  0.05    1.0  0.0

## 3.9 常见坑与下一章衔接

- 坑 1：没有先按 `code, date` 排序就做 `pct_change` 或 `rolling`。结果会把时间顺序打乱。
- 坑 2：宽表相乘前没有对齐索引和列。pandas 会自动按标签对齐，但如果日期或代码缺失，结果可能出现大量 `NaN`。
- 坑 3：用今天收盘后的信号买今天的收益。后续回测一律用 `shift(1)` 让权重滞后一日生效。

本章先直接使用了 `data/sample/` 的缓存。第 04 章会回到数据源头：从 AKShare 获取 ETF 日线，统一字段为 `date/code/open/high/low/close/volume/amount`，并把同样格式的数据缓存到本地。


In [19]:
chapter03_objects = pd.Series(
    {
        "features_rows": len(features),
        "close_matrix_shape": close_matrix.shape,
        "returns_matrix_shape": returns_matrix.shape,
        "score_matrix_shape": score_matrix.shape,
        "weights_matrix_shape": weights_matrix.shape,
        "latest_score_date": latest_score_date.date(),
    }
)

chapter03_objects.to_frame("value")


,value
features_rows,2900
close_matrix_shape,"(725, 4)"
returns_matrix_shape,"(725, 4)"
score_matrix_shape,"(704, 4)"
weights_matrix_shape,"(704, 4)"
latest_score_date,2023-12-29
